In [2]:
from langchain_community.document_loaders import (PyPDFLoader,
                                                  TextLoader,
                                                  CSVLoader) 
from pathlib import Path


def load_all_documents(data_dir="sample_data"):
    """Load all documents from multiple sources"""

    all_docs = []

    for txt_file in Path(data_dir).glob("*.txt"):
        loader = TextLoader(str(txt_file))
        all_docs.extend(loader.load())
        print(f"Loaded: {txt_file.name}")

    for csv_file in Path(data_dir).glob("*.csv"):
        loader = CSVLoader(str(csv_file))
        all_docs.extend(loader.load())
        print(f"Loaded: {csv_file.name}")

    return all_docs

docs = load_all_documents()
print(f"\n Total docs: {len(docs)}")

Loaded: notes.txt
Loaded: products.csv

 Total docs: 16


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print(f"Splitting docs")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=200)

chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")


Splitting docs
Created 27 chunks


In [4]:
from langchain_community.vectorstores import FAISS 
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
index_path = "./rag_vectorstore"
if Path(index_path).exists():
    print("Loading the existing vector store")
    vectorstore = FAISS.load_local(index_path,
                                   embeddings,
                                   allow_dangerous_deserialization=True)

else:
    print("Creating new vector store (this may take a minute)...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(index_path)

print("Vector store ready")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Creating new vector store (this may take a minute)...
Vector store ready


In [5]:
retriever = vectorstore.as_retriever(search_type="similarity",
                                     search_kwargs={"k":4})
print("Retriever created")

Retriever created


In [6]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant. Answer the question based on the context below.
If you cannot answer based on the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
print("Prompt template created")


Prompt template created


In [7]:
from langchain_groq import ChatGroq 

llm = ChatGroq(model="llama-3.3-70b-versatile",
               temperature=0.0)

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Helper function to format documents
def format_docs(docs):
    """Format retrieved documents into a single string"""
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context":retriever | format_docs,
        "question":RunnablePassthrough()
    }
    | prompt 
    | llm 
    | StrOutputParser()
)    

In [9]:
questions = [
    "What is RAG?",
    "What are the recommended chunk sizes?",
    "What embedding models are available?"
]

for question in questions:
    print("\n" + "="*70)
    print(f"Question: {question}")
    print("="*70)

    ans = rag_chain.invoke(question)
    print(f"Answer:\n{ans}")

    print("\nSource Documents:")
    docs = retriever.invoke(question)
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        print(f"  {i}. {Path(source).name}")


Question: What is RAG?
Answer:
RAG = Retrieval + Context + Generation.

Source Documents:
  1. notes.txt
  2. notes.txt
  3. notes.txt
  4. notes.txt

Question: What are the recommended chunk sizes?
Answer:
The recommended chunk sizes vary based on the content type. Here are the guidelines:

- General Text: 1000 characters
- Technical Docs: 500-800 characters
- Source Code: 200-400 characters
- Long Articles: 1500-2000 characters
- Conversational Data: 100-200 characters

Additionally, the overlap sizes are also recommended as follows:
- General Text: 200 characters
- Technical Docs: 100-150 characters
- Source Code: 50-100 characters
- Long Articles: 300 characters
- Conversational Data: 20 characters

Source Documents:
  1. notes.txt
  2. notes.txt
  3. notes.txt
  4. notes.txt

Question: What embedding models are available?
Answer:
There are several embedding models available, including:

1. OpenAI text-embedding-3-small: 1536 dimensions, $0.00002 per 1K tokens, MTEB Score: 62.3%
2